# Regenerating the TransferSemi figures in Python

This notebook reproduces, in pure Python (pandas/matplotlib/seaborn), the 14
standalone PDFs listed in `README.md` Section 7 under
`outputs/figures_previous_results/`. It is a line-by-line port of the R/ggplot2
code in `visualization_github/*.Rmd` — same raw data, same filtering, same
grouping, same aggregation. Figures are displayed inline with `plt.show()`;
**no PDFs are written**.

## How to run this
Pick a kernel that has `pandas`/`numpy`/`matplotlib`/`seaborn` — e.g. this
repo's own environment, or (confirmed to work on this cluster)
`/blue/muxuan.liang/yongzhuo/.conda/envs/bolero_stats/bin/python`.

## Why Step 1 shells out to R
Python cannot read R's `.rds`/`.RData` binary serialization format directly.
Step 1 below runs a small R script
(`visualization_github/python_port/export_raw_data.R`) that parses the raw
simulation/real-data files with the *exact same* logic as the Rmd notebooks
and writes tidy CSVs to `outputs/python_notebook_data/`. Everything after
that step is pure Python — no `rpy2`, no R required at import time.

## Known, faithfully-reproduced quirks in the original R code
Porting this surfaced two real inconsistencies in the source `.Rmd` files.
Per your instruction this notebook **copies the R logic as-is**, bugs
included, rather than silently fixing them:

1. **DR's outlier pre-filter uses `Q3` instead of the upper whisker.**
   In `sim_visualization_DR.Rmd` / `MSE_visualization.Rmd`, the CATE panels
   trim points outside `[lower_whisker, upper_whisker]` (`boxplot.stats()$stats[5]`)
   before drawing the final box, but the DR panels trim to
   `[lower_whisker, Q3]` (`stats[4]`, not `stats[5]`) — discarding the entire
   top quartile of every group, not just true outliers. This notebook keeps
   that asymmetry for DR and the whisker-based trim for CATE.
2. **`display_method()`'s regex for "Late Fusion" / "Late Fusion with Nuisance"
   matches both the tuned and untuned PLM variants** (`^Para_Transfer` matches
   `Para_Transfer` *and* `Para_Transfer_Tuned`; same for `NonPara_Transfer`).
   This only affects the task-comparison and negative-transfer sections
   (which use `display_method()`); the quantile-based Mean/Max MSE panels
   pre-filter to the `_Tuned` rows before recoding, so they are unaffected.

## Scope / visual-fidelity notes
- ggplot2's `facet_grid(scales = "free_y")` frees the y-axis **per row**;
  for simplicity this port frees the y-axis **per subplot**. Relative
  comparisons *within* each panel are preserved; only exact axis ranges
  differ from the R output.
- `"steelblue1"` (used for "Representation Transfer") is approximated as
  `#63B8FF`.
- The real-data correlation heatmap (`corr.pdf`) in the R source computes
  correlations twice (once over all columns, once over categorical-only
  columns) and `rbind()`s both into the same heatmap data — but the two
  passes give numerically identical correlations for shared variables, so
  the duplication has no visible effect. This port computes each
  (model, variable) correlation once, which reproduces the same rendered
  heatmap more directly.


## Step 0 — setup: paths, constants, and shared plotting helpers

In [ ]:
import os
import re
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

REPO_ROOT = "/home/chen.yongzhuo/blue_gator_zhang/chen.yongzhuo/TransferSemi_github"
DATA_DIR = os.path.join(REPO_ROOT, "outputs", "python_notebook_data")

METHOD_ORDER = [
    "Baseline", "Data Pooling", "Oracle Trans-Lasso", "Representation Transfer",
    "pERM", "ARMUL", "Late Fusion", "Late Fusion with Nuisance",
]
COLORS = {
    "Baseline": "#FFB3BA",
    "Data Pooling": "#FFDFBA",
    "Oracle Trans-Lasso": "#BAE1FF",
    "pERM": "#9BF6FF",
    "Representation Transfer": "#63B8FF",  # R\'s "steelblue1"
    "ARMUL": "#6FA8DC",
    "Late Fusion": "#7CFC00",
    "Late Fusion with Nuisance": "#2ECC71",
}
SM_SIMILARITY = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
MAIN_SIMILARITY = [0, 0.4, 0.8, 1.0]
N_VALUES = [100, 200]
WHITE_RED = LinearSegmentedColormap.from_list("white_red", ["white", "red"])


def display_method(method):
    """Port of display_method() in MSE_visualization.Rmd (lines 1269-1281)."""
    s = str(method)
    if re.match(r"^Baseline", s, re.I):
        return "Baseline"
    if re.match(r"^Pooled", s, re.I) or re.search(r"^Data Pooling", s, re.I):
        return "Data Pooling"
    if re.match(r"^Sai_Transfer", s, re.I) or re.search(r"Oracle Trans", s, re.I):
        return "Oracle Trans-Lasso"
    if re.match(r"^NonPara_Transfer", s, re.I) or re.search(r"Late Fusion with Nuisance", s, re.I):
        return "Late Fusion with Nuisance"
    if re.match(r"^Para_Transfer", s, re.I) or re.fullmatch(r"Late Fusion", s, re.I):
        return "Late Fusion"
    if re.match(r"^Net_Transfer", s, re.I) or re.search(r"Representation Transfer", s, re.I):
        return "Representation Transfer"
    if re.match(r"^pERM", s, re.I):
        return "pERM"
    if re.match(r"^ARMUL", s, re.I):
        return "ARMUL"
    return s


def r_five_number_summary(values, coef):
    """Port of base R boxplot.stats(x, coef=coef)$stats."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return None
    q1, med, q3 = np.percentile(values, [25, 50, 75])
    iqr = q3 - q1
    lo_fence, hi_fence = q1 - coef * iqr, q3 + coef * iqr
    within = values[(values >= lo_fence) & (values <= hi_fence)]
    whislo = within.min() if len(within) else q1
    whishi = within.max() if len(within) else q3
    return whislo, q1, med, q3, whishi


def ggplot_box_stats(values, coef):
    """What geom_boxplot(outlier.shape=NA, coef=coef) actually draws."""
    s = r_five_number_summary(values, coef)
    if s is None:
        return None
    whislo, q1, med, q3, whishi = s
    return dict(whislo=whislo, q1=q1, med=med, q3=q3, whishi=whishi, fliers=[])


def draw_method_box(ax, values, x, color, width=0.8, coef=2):
    stats = ggplot_box_stats(values, coef=coef)
    if stats is None:
        return
    stats["label"] = ""
    bp = ax.bxp([stats], positions=[x], widths=width, showfliers=False, patch_artist=True)
    for patch in bp["boxes"]:
        patch.set_facecolor(color)
        patch.set_edgecolor("black")
        patch.set_linewidth(0.6)
    for med_line in bp["medians"]:
        med_line.set_color("black")


def method_legend(fig, methods, ncol=4, loc="outside lower center"):
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=COLORS.get(m, "grey"), edgecolor="black") for m in methods]
    fig.legend(handles, methods, loc=loc, ncol=ncol, frameon=False, fontsize=9)


def outlier_trim(df, group_cols, value_col, coef, upper_index):
    """Port of the two-stage trim: boxplot.stats(coef=coef)$stats[c(1, upper_index)]
    defines a keep-window; rows outside it are dropped before the final box is drawn.
    upper_index: 5 = upper whisker (CATE), 4 = Q3 (DR -- see the note in the intro)."""
    kept = []
    for _, g in df.groupby(group_cols):
        s = r_five_number_summary(g[value_col].values, coef=coef)
        if s is None:
            continue
        lower, upper = s[0], s[upper_index - 1]
        kept.append(g[(g[value_col] >= lower) & (g[value_col] <= upper)])
    return pd.concat(kept, ignore_index=True) if kept else df.iloc[0:0]


print("Setup OK. REPO_ROOT =", REPO_ROOT)


## Step 1 — export the raw R data to CSV

Runs `visualization_github/python_port/export_raw_data.R` (if the CSVs
aren't already there) to parse the raw `.rds`/`.RData` simulation output
with R, using the same directory/file patterns as the Rmd notebooks. This
step needs an R module (`module load R/4.3` on this cluster); everything
after it is pure Python.


In [ ]:
plm_csv = os.path.join(DATA_DIR, "plm_raw.csv")
if not os.path.exists(plm_csv):
    export_script = os.path.join(REPO_ROOT, "visualization_github", "python_port", "export_raw_data.R")
    cmd = f"source /etc/profile.d/*.sh 2>/dev/null; module load R/4.3; Rscript {export_script}"
    result = subprocess.run(["bash", "-c", cmd], cwd=REPO_ROOT, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("export_raw_data.R failed")
else:
    print("CSVs already present in", DATA_DIR, "- skipping R export.")


## Step 2 — load the raw data into pandas

In [ ]:
plm = pd.read_csv(f"{DATA_DIR}/plm_raw.csv")
cate = pd.read_csv(f"{DATA_DIR}/cate_raw.csv")
dr = pd.read_csv(f"{DATA_DIR}/dr_raw.csv")
finetune_implemented_mse = pd.read_csv(f"{DATA_DIR}/finetune_implemented_mse.csv")
finetune_global_mse = pd.read_csv(f"{DATA_DIR}/finetune_global_mse.csv")
finetune_implemented_runtime = pd.read_csv(f"{DATA_DIR}/finetune_implemented_runtime.csv")
finetune_global_runtime = pd.read_csv(f"{DATA_DIR}/finetune_global_runtime.csv")
real_data_raw = pd.read_csv(f"{DATA_DIR}/real_data_raw.csv")
real_corr_all = pd.read_csv(f"{DATA_DIR}/real_corr_all_columns.csv")

print("plm:", plm.shape, " cate:", cate.shape, " dr:", dr.shape)
print("finetune implemented/global MSE:", finetune_implemented_mse.shape, finetune_global_mse.shape)
print("real_data_raw:", real_data_raw.shape, " real_corr_all:", real_corr_all.shape)


## Figure 1 — `PLM_heat_map_PLM.pdf`

Port of `PLM_Visualization_PLM.Rmd` (final chunk, ~line 688): log-transform
Task1-5, average over seeds grouped by (method, similarity, n), keep n=200
only, facet by method (all 8), x=task, y=similarity.


In [ ]:
def heatmap_long(df, task_prefix, methods):
    d = df.copy()
    task_cols = [f"{task_prefix}{i}" for i in range(1, 6)]
    method_col = "method" if "method" in d.columns else "Method"
    d["Method"] = d[method_col].apply(display_method)
    d = d[np.isfinite(d[task_cols]).all(axis=1)]
    d[task_cols] = np.log(d[task_cols])
    grouped = d.groupby(["Method", "similarity", "n"])[task_cols].mean().reset_index()
    grouped = grouped[grouped["n"] == 200]
    long = grouped.melt(id_vars=["Method", "similarity", "n"], value_vars=task_cols,
                         var_name="task", value_name="value")
    long["task"] = long["task"].str.replace(task_prefix, "Task", regex=False)
    return long[long["Method"].isin(methods)]


def draw_heatmap_grid(long_df, methods, ncols, title):
    nrow = int(np.ceil(len(methods) / ncols))
    fig, axes = plt.subplots(nrow, ncols, figsize=(3.2 * ncols, 2.6 * nrow), squeeze=False, constrained_layout=True)
    vmin, vmax = long_df["value"].min(), long_df["value"].max()
    x_levels = sorted(long_df["task"].unique())
    y_levels = sorted(long_df["similarity"].unique())
    im = None
    for i, m in enumerate(methods):
        ax = axes[i // ncols, i % ncols]
        sub = long_df[long_df["Method"] == m]
        grid = sub.pivot(index="similarity", columns="task", values="value").reindex(
            index=y_levels, columns=x_levels
        )
        im = ax.imshow(grid.values, cmap=WHITE_RED, vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(x_levels))); ax.set_xticklabels(x_levels, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(y_levels))); ax.set_yticklabels(y_levels, fontsize=8)
        ax.set_title(m, fontsize=10)
    for j in range(len(methods), nrow * ncols):
        axes[j // ncols, j % ncols].axis("off")
    fig.suptitle(title, fontsize=13)
    fig.colorbar(im, ax=axes, shrink=0.6, label="Mean of Log")
    plt.show()


plm_heat_methods = METHOD_ORDER  # all 8, matches PLM_Visualization_PLM.Rmd
plm_heat_long = heatmap_long(plm, "task", plm_heat_methods)
draw_heatmap_grid(plm_heat_long, plm_heat_methods, ncols=4, title="PLM_heat_map_PLM")


## Figure 2 — `CATE_heat_map_CATE.pdf`

Port of `sim_visualization_CATE.Rmd` (final chunk, ~line 449): same recipe,
restricted to 6 methods (`pERM` and `Representation Transfer` excluded).


In [ ]:
cate_heat_methods = ["Baseline", "Data Pooling", "ARMUL", "Oracle Trans-Lasso",
                      "Late Fusion", "Late Fusion with Nuisance"]
cate_heat_long = heatmap_long(cate, "Task", cate_heat_methods)
draw_heatmap_grid(cate_heat_long, cate_heat_methods, ncols=3, title="CATE_heat_map_CATE")


## Figure 3 — `DR_heat_map_DR.pdf`

Port of `sim_visualization_DR.Rmd` (final chunk, ~line 416): same recipe,
restricted to 6 methods (`Representation Transfer` doesn't exist for DR;
`Late Fusion with Nuisance` doesn't either — DR has no semiparametric variant).


In [ ]:
dr_heat_methods = ["Baseline", "Oracle Trans-Lasso", "Late Fusion", "Data Pooling", "ARMUL", "pERM"]
dr_heat_long = heatmap_long(dr, "Task", dr_heat_methods)
draw_heatmap_grid(dr_heat_long, dr_heat_methods, ncols=3, title="DR_heat_map_DR")


## Figures 4 & 5 — `MSE_Mean_combined_plot_sm.pdf` and `MSE_max_combined_plot_sm.pdf`

Port of `MSE_visualization.Rmd`. Each is three independently-faceted
(n x similarity) panels side by side (PLM | DR | CATE) sharing one legend.

For **PLM**, each box is *not* a boxplot of raw per-seed values — it's a
boxplot of the four quantile points `quantile(mean_value, c(0,.25,.5,.75))`
computed per (method, n, similarity) first (`prepare_data()`/`prepare_data_()`
in the R source). For **CATE/DR**, each box is a boxplot of per-seed
mean/max-across-tasks MSE, after the two-stage outlier trim described in the
intro (note the DR Q3-vs-whisker asymmetry).


In [ ]:
PLM_QUANTILE_METHODS = {
    "Baseline": "Baseline", "Sai_Transfer": "Oracle Trans-Lasso",
    "Para_Transfer_Tuned": "Late Fusion", "NonPara_Transfer_Tuned": "Late Fusion with Nuisance",
    "Net_Transfer": "Representation Transfer", "ARMUL_transfer": "ARMUL",
    "pERM_Transfer": "pERM", "Pooled": "Data Pooling",
}


def plm_quantile_long(value_col):
    rows = []
    for n in N_VALUES:
        for sim in SM_SIMILARITY:
            subset = plm[(plm["n"] == n) & (np.isclose(plm["similarity"], sim))]
            for raw_method, disp in PLM_QUANTILE_METHODS.items():
                vals = subset.loc[subset["method"] == raw_method, value_col].values
                if len(vals) == 0:
                    continue
                for q in np.percentile(vals, [0, 25, 50, 75]):
                    rows.append({"n": n, "similarity": sim, "Method": disp, "Value": q})
    return pd.DataFrame(rows)


def cate_dr_panel(df, agg, exclude_methods, outlier_upper_index, coef_trim):
    d = df.copy()
    task_cols = [c for c in d.columns if c.lower().startswith("task")]
    d["Value"] = d[task_cols].mean(axis=1) if agg == "mean" else d[task_cols].max(axis=1)
    d["Method"] = d["Method"].apply(display_method)
    d = d[~d["Method"].isin(exclude_methods)]
    return outlier_trim(d, ["Method", "n", "similarity"], "Value", coef=coef_trim, upper_index=outlier_upper_index)


def draw_full_facet_panel(subfig, df, methods, title, ylim=None, coef=2):
    axes = subfig.subplots(len(N_VALUES), len(SM_SIMILARITY), squeeze=False)
    for r, n in enumerate(N_VALUES):
        for c, sim in enumerate(SM_SIMILARITY):
            ax = axes[r, c]
            cell = df[(df["n"] == n) & (np.isclose(df["similarity"], sim))]
            for i, m in enumerate(methods):
                vals = cell.loc[cell["Method"] == m, "Value"].values
                draw_method_box(ax, vals, i, COLORS.get(m, "grey"), width=0.8, coef=coef)
            ax.set_yscale("log")
            ax.set_xlim(-0.6, len(methods) - 0.4)
            ax.set_xticks([])
            if ylim is not None:
                ax.set_ylim(*ylim)
            if r == 0:
                ax.set_title(f"sim={sim}", fontsize=7)
            if c == 0:
                ax.set_ylabel(f"n={n}", fontsize=8)
    subfig.suptitle(title, fontsize=11)


def mse_combined_figure(plm_long, dr_df, cate_df, dr_ylim, plm_ylim, coef, suptitle):
    fig = plt.figure(figsize=(21, 6), constrained_layout=True)
    sub_plm, sub_dr, sub_cate = fig.subfigures(1, 3)
    draw_full_facet_panel(sub_plm, plm_long, METHOD_ORDER, "PLM", ylim=plm_ylim, coef=coef)
    draw_full_facet_panel(sub_dr, dr_df, METHOD_ORDER, "DR", ylim=dr_ylim, coef=coef)
    draw_full_facet_panel(sub_cate, cate_df, METHOD_ORDER, "CATE", ylim=None, coef=coef)
    fig.suptitle(suptitle, fontsize=14, y=1.03)
    method_legend(fig, METHOD_ORDER, ncol=8)
    plt.show()


# --- Mean MSE combined (MSE_Mean_combined_plot_sm.pdf) ---
plm_mean_long = plm_quantile_long("mean_value")
dr_mean = cate_dr_panel(dr, "mean", exclude_methods=[], outlier_upper_index=4, coef_trim=1)
cate_mean = cate_dr_panel(cate, "mean", exclude_methods=["Representation Transfer"],
                           outlier_upper_index=5, coef_trim=1)
mse_combined_figure(plm_mean_long, dr_mean, cate_mean,
                     dr_ylim=(0.025, 0.3), plm_ylim=(1e-5, None), coef=2,
                     suptitle="MSE_Mean_combined_plot_sm")

# --- Max MSE combined (MSE_max_combined_plot_sm.pdf) ---
plm_max_long = plm_quantile_long("max_value")
dr_max = cate_dr_panel(dr, "max", exclude_methods=[], outlier_upper_index=4, coef_trim=1)
cate_max = cate_dr_panel(cate, "max", exclude_methods=["Representation Transfer"],
                          outlier_upper_index=5, coef_trim=1.5)
mse_combined_figure(plm_max_long, dr_max, cate_max,
                     dr_ylim=None, plm_ylim=(5e-5, None), coef=2,
                     suptitle="MSE_max_combined_plot_sm")


## Figures 6-8 — `PLM/DR/CATE_Task_Comparison_with_legend.pdf`

Port of the `task-level-comparison` chunk in `MSE_visualization.Rmd`
(~line 1332): raw per-seed Task1/Task4/Task5 values (not mean/max-aggregated),
restricted to n in {100,200} and similarity in {0, 0.4, 0.8, 1.0}, outlier-trimmed
per (n, Task, similarity, Method) group with `coef=1.5`, dodged boxplot with
Method dodged within each similarity x-position, faceted n (rows) x Task (cols).

Note: because `display_method()` maps both `Para_Transfer` and
`Para_Transfer_Tuned` to `"Late Fusion"` (see the intro's quirk #2), the PLM
panel's "Late Fusion" box here pools the tuned *and* untuned estimator —
this matches the R source's behavior for this section exactly.


In [ ]:
def task_comparison_data(df, model_name, exclude_methods=()):
    d = df.copy()
    d.columns = [c.lower() for c in d.columns]
    method_col = "method"
    d["Method"] = d[method_col].apply(display_method)
    long = d.melt(id_vars=["n", "similarity", "seed", "Method"], value_vars=["task1", "task4", "task5"],
                  var_name="Task", value_name="Value")
    long = long[long["similarity"].apply(lambda s: min(abs(s - x) for x in MAIN_SIMILARITY) < 1e-9)]
    long = long[(long["Value"] > 0) & np.isfinite(long["Value"])]
    long = long[long["Method"].isin(METHOD_ORDER) & ~long["Method"].isin(exclude_methods)]
    long["Model"] = model_name
    return outlier_trim(long, ["n", "Task", "similarity", "Method"], "Value", coef=1.5, upper_index=5)


def draw_task_comparison(df, title):
    tasks = ["task1", "task4", "task5"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), squeeze=False, constrained_layout=True)
    width = 0.9 / len(METHOD_ORDER)
    for row, n in enumerate(N_VALUES):
        for col, task in enumerate(tasks):
            ax = axes[row, col]
            sub = df[(df["n"] == n) & (df["Task"] == task)]
            for si, sim in enumerate(MAIN_SIMILARITY):
                for mi, m in enumerate(METHOD_ORDER):
                    vals = sub.loc[(np.isclose(sub["similarity"], sim)) & (sub["Method"] == m), "Value"].values
                    x = si + (mi - (len(METHOD_ORDER) - 1) / 2) * width
                    draw_method_box(ax, vals, x, COLORS.get(m, "grey"), width=width * 0.9, coef=1.5)
            ax.set_yscale("log")
            ax.set_xticks(range(len(MAIN_SIMILARITY)))
            ax.set_xticklabels(MAIN_SIMILARITY)
            ax.set_xlabel("Similarity")
            if row == 0:
                ax.set_title(f"Task {task[-1]}")
            if col == 0:
                ax.set_ylabel(f"n={n}\nMSE")
    fig.suptitle(title, fontsize=14)
    method_legend(fig, METHOD_ORDER, ncol=8)
    plt.show()


plm_task_cmp = task_comparison_data(plm, "PLM")
draw_task_comparison(plm_task_cmp, "PLM_Task_Comparison_with_legend")

dr_task_cmp = task_comparison_data(dr, "DR")
draw_task_comparison(dr_task_cmp, "DR_Task_Comparison_with_legend")

cate_task_cmp = task_comparison_data(cate, "CATE", exclude_methods=["Representation Transfer"])
draw_task_comparison(cate_task_cmp, "CATE_Task_Comparison_with_legend")


## Figures 9-11 — `PLM/CAT/DR_task1_task4_task5_negative_transfer_all_methods.pdf`

Port of the `task-level-negative-transfer` / `task145-negative-transfer`
chunks (~lines 1474-1744): for each (model, n, similarity, seed, Task, method),
pair each non-baseline method's MSE against that exact seed's Baseline MSE;
`Negative_transfer_rate` = fraction of seeds where the method did *worse*
than Baseline, restricted to Task 1/4/5 and the full similarity grid.


In [ ]:
METHOD_ORDER_NT = ["Data Pooling", "Oracle Trans-Lasso", "pERM", "Representation Transfer",
                    "ARMUL", "Late Fusion", "Late Fusion with Nuisance"]


def negative_transfer_rate(df, model_name):
    d = df.copy()
    d.columns = [c.lower() for c in d.columns]
    d["Method"] = d["method"].apply(display_method)
    long = d.melt(id_vars=["n", "similarity", "seed", "Method"], value_vars=["task1", "task4", "task5"],
                  var_name="Task", value_name="MSE")
    long = long[long["Method"].isin(["Baseline"] + METHOD_ORDER_NT)]
    long = long[np.isfinite(long["MSE"])]
    seed_level = long.groupby(["n", "similarity", "seed", "Task", "Method"], as_index=False)["MSE"].mean()
    baseline = seed_level[seed_level["Method"] == "Baseline"][["n", "similarity", "seed", "Task", "MSE"]]
    baseline = baseline.rename(columns={"MSE": "Baseline_MSE"})
    paired = seed_level[seed_level["Method"] != "Baseline"].merge(
        baseline, on=["n", "similarity", "seed", "Task"], how="inner")
    paired["worse"] = paired["MSE"] > paired["Baseline_MSE"]
    rate = paired.groupby(["n", "similarity", "Task", "Method"], as_index=False)["worse"].mean()
    return rate.rename(columns={"worse": "Negative_transfer_rate"})


def draw_negative_transfer(rate, title):
    tasks = ["task1", "task4", "task5"]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7), squeeze=False, constrained_layout=True)
    for row, n in enumerate(N_VALUES):
        for col, task in enumerate(tasks):
            ax = axes[row, col]
            sub = rate[(rate["Task"] == task) & (rate["n"] == n)]
            for m in METHOD_ORDER_NT:
                g = sub[sub["Method"] == m].sort_values("similarity")
                if g.empty:
                    continue
                ax.plot(g["similarity"], g["Negative_transfer_rate"], marker="o",
                        label=m, color=COLORS.get(m, "grey"), linewidth=1.3, markersize=4)
            ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.7)
            ax.set_ylim(0, 1)
            ax.set_xticks(SM_SIMILARITY)
            ax.set_xlabel("Similarity")
            if row == 0:
                ax.set_title(f"Task {task[-1]}")
            if col == 0:
                ax.set_ylabel(f"n={n}\nProportion worse than baseline")
    fig.suptitle(title, fontsize=14)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncol=4, fontsize=9, frameon=False)
    plt.show()


draw_negative_transfer(negative_transfer_rate(plm, "PLM"), "PLM_task1_task4_task5_negative_transfer_all_methods")
draw_negative_transfer(negative_transfer_rate(cate, "CATE"), "CAT_task1_task4_task5_negative_transfer_all_methods")
draw_negative_transfer(negative_transfer_rate(dr, "DR"), "DR_task1_task4_task5_negative_transfer_all_methods")


## Figure 12 — `tuning_strategy_compare.pdf`

Port of `finetune_strategy_comparison.Rmd`. Compares the `NonPara_Transfer_Tuned`
("Late Fusion with Nuisance") estimator's mean MSE and wall-clock runtime under
two fine-tuning strategies: the repo-default "Implemented" strategy
(`results_5_dim8` for MSE, `results_11_dim8`'s timing folders for runtime) vs.
a "Global Fine-tuned" variant (`results_finetune_new8` / `_new_time8`). The
third "more parameters" strategy needs `results_6_dim8`/`time_6_dim_8`, which
this checkout doesn't include (per the README), so only two strategies appear
here — matching what the previous-results render actually produces.


In [ ]:
strategy_colors = {
    "Global Fine-tuned Late Fusion with Nuisance": "#F8766D",
    "Late Fusion with Nuisance": "#00BA38",
}
runtime_colors = {"Global Fine-tuned": "#F8766D", "Implemented": "#00BA38"}

mse_comparison = pd.concat([
    finetune_global_mse.assign(Method=finetune_global_mse["Method"]),
    finetune_implemented_mse.assign(Method=finetune_implemented_mse["Method"]),
], ignore_index=True)
runtime_comparison = pd.concat([finetune_global_runtime, finetune_implemented_runtime], ignore_index=True)

fig, axes = plt.subplots(2, len(SM_SIMILARITY), figsize=(18, 6), squeeze=False, sharey=False, constrained_layout=True)
mse_methods = list(strategy_colors.keys())
for r, n in enumerate(N_VALUES):
    for c, sim in enumerate(SM_SIMILARITY):
        ax = axes[r, c]
        cell = mse_comparison[(mse_comparison["n"] == n) & (np.isclose(mse_comparison["similarity"], sim))]
        for i, m in enumerate(mse_methods):
            vals = cell.loc[cell["Method"] == m, "mean_mse"].values
            draw_method_box(ax, vals, i, strategy_colors[m], width=0.7, coef=2)
        ax.set_yscale("log")
        ax.set_xticks([])
        if r == 0:
            ax.set_title(f"sim={sim}", fontsize=8)
        if c == 0:
            ax.set_ylabel(f"n={n}\nMean MSE")
fig.suptitle("tuning_strategy_compare — MSE panel", fontsize=13)
method_legend(fig, mse_methods, ncol=2)
plt.show()

fig, axes = plt.subplots(2, len(SM_SIMILARITY), figsize=(18, 6), squeeze=False, sharey=False, constrained_layout=True)
rt_methods = list(runtime_colors.keys())
for r, n in enumerate(N_VALUES):
    for c, sim in enumerate(SM_SIMILARITY):
        ax = axes[r, c]
        cell = runtime_comparison[(runtime_comparison["n"] == n) & (np.isclose(runtime_comparison["similarity"], sim))]
        for i, m in enumerate(rt_methods):
            vals = cell.loc[cell["Method"] == m, "runtime_minutes"].values
            if len(vals) == 0:
                continue
            med = np.median(vals)
            q1, q3 = np.percentile(vals, [25, 75])
            lo, hi = max(vals.min(), q1 - 1.5 * (q3 - q1)), min(vals.max(), q3 + 1.5 * (q3 - q1))
            ax.bar(i, med, width=0.6, color=runtime_colors[m])
            ax.errorbar([i], [med], yerr=[[med - lo], [hi - med]], fmt="none", ecolor="black", capsize=3)
        ax.set_yscale("log")
        ax.set_xticks([])
        if r == 0:
            ax.set_title(f"sim={sim}", fontsize=8)
        if c == 0:
            ax.set_ylabel(f"n={n}\nTime (min)")
fig.suptitle("tuning_strategy_compare — runtime panel", fontsize=13)
method_legend(fig, rt_methods, ncol=2)
plt.show()


## Figure 13 — `corr.pdf` (real-data correlation heatmap)

Port of `read_real_analysis.Rmd` (final heatmap chunk, ~line 704): for each
model and covariate, the Pearson correlation between the covariate and the
model's fitted CATE, averaged across the 4 pseudo-data tasks. See the intro
note about the harmless categorical/all-columns duplication in the source.


In [ ]:
corr_agg = real_corr_all.groupby(["model", "Variable"], as_index=False)["Correlation"].mean()
rename_model = {
    "no_transfer": "Baseline", "para_transfer_tuned": "Late Fusion", "Sai_transfer": "Oracle Trans-Lasso",
    "semipara_transfer_tuned": "Late Fusion with Nuisance", "pool": "Data Pooling",
    "ARMUL": "ARMUL", "pERM": "pERM",
}
corr_agg["Model"] = corr_agg["model"].map(rename_model)
corr_agg = corr_agg.dropna(subset=["Model"])
rank_order = (corr_agg[corr_agg["Model"] == "Late Fusion with Nuisance"]
              .sort_values("Correlation", ascending=False)["Variable"].tolist())
model_order = ["Late Fusion with Nuisance", "Late Fusion", "ARMUL", "pERM",
               "Oracle Trans-Lasso", "Data Pooling", "Baseline"]
pivot = corr_agg.pivot(index="Model", columns="Variable", values="Correlation").reindex(
    index=[m for m in model_order if m in corr_agg["Model"].unique()], columns=rank_order
)

fig, ax = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
lim = np.nanmax(np.abs(pivot.values))
im = ax.imshow(pivot.values, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index, fontsize=9)
fig.colorbar(im, ax=ax, label="Correlation", shrink=0.8)
fig.suptitle("corr.pdf")
plt.show()


## Figure 14 — `combined_err.pdf` (real-data mean MSE +/- CI)

Port of `read_real_analysis.Rmd` (~line 228): mean pseudo-data MSE per
method with a 95%-style error bar (`1.96 * sd / 10`, matching the R source's
exact — unusually small — denominator).


In [ ]:
mean_sd = real_data_raw.groupby("Model", as_index=False)["MSE"].agg(["mean", "std"]).reset_index()
keep_models = {
    "eval_no_transfer": "Baseline", "eval_pooling": "Data Pooling", "eval_Sai": "Oracle Trans-Lasso",
    "eval_pERM": "pERM", "eval_representation_learning": "Representation Transfer",
    "eval_ARMUL": "ARMUL", "eval_para_transfer_tuned": "Late Fusion",
    "eval_semipara_transfer_tuned": "Late Fusion with Nuisance", "eval_pool": "Data Pooling",
}
mean_sd["Method"] = mean_sd["Model"].map(keep_models)
mean_sd = mean_sd.dropna(subset=["Method"])
plot_order = ["Baseline", "Data Pooling", "Oracle Trans-Lasso", "pERM",
              "Representation Transfer", "ARMUL", "Late Fusion", "Late Fusion with Nuisance"]
mean_sd["Method"] = pd.Categorical(mean_sd["Method"], categories=plot_order, ordered=True)
mean_sd = mean_sd.sort_values("Method").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
for i, row in mean_sd.iterrows():
    err = 1.96 * row["std"] / 10
    ax.errorbar([i], [row["mean"]], yerr=[[err], [err]], fmt="o", markersize=7,
                color=COLORS.get(row["Method"], "grey"), capsize=4)
ax.set_xticks(range(len(mean_sd)))
ax.set_xticklabels([])
ax.set_ylabel("Mean MSE")
method_legend(fig, [m for m in plot_order if m in mean_sd["Method"].values], ncol=4)
fig.suptitle("combined_err.pdf")
plt.show()
